In [0]:
import requests
import json
import uuid
import hashlib
from datetime import date
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from delta.tables import DeltaTable

# API Key
JSEARCH_API_KEY = dbutils.secrets.get("jobs_automation", "jsearch")

def fetch_and_store_jobs(search_keyword):
    today_str = str(date.today())
    
    print(f"🔍 Searching for: '{search_keyword}'")
    
    # 1. SMART CHECK: Eroju already ee keyword tho jobs unnayemo check chestunnam
    check_query = f"""
        SELECT * FROM jobs_automation_db.default.raw_jobs_staging 
        WHERE search_keyword = '{search_keyword}' AND fetch_date = '{today_str}'
    """
    existing_jobs_df = spark.sql(check_query)
    
    if existing_jobs_df.count() > 0:
        print(f"♻️ API LIMIT SAVED! Eroju already '{search_keyword}' meeda jobs fetch ayyayi. Database nundi load chestunnam...")
        return existing_jobs_df
        
    print(f"🌐 Kotha search! JSearch API nundi '{search_keyword}' 50+ jobs fetch chestunnam...")
    
    url = "https://jsearch.p.rapidapi.com/search-v2"
    querystring = {"query": search_keyword, "page": "1", "num_pages": "5"}
    headers = {
        "x-rapidapi-key": JSEARCH_API_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    
    if response.status_code == 200:
        json_resp = response.json()
        jobs_list = []
        
        # Mana bulletproof JSON extraction logic
        if isinstance(json_resp, dict):
            if 'data' in json_resp:
                inner_data = json_resp['data']
                if isinstance(inner_data, list):
                     jobs_list = inner_data
                elif isinstance(inner_data, dict) and 'jobs' in inner_data and isinstance(inner_data['jobs'], list):
                     jobs_list = inner_data['jobs']
            elif 'jobs' in json_resp and isinstance(json_resp['jobs'], list):
                 jobs_list = json_resp['jobs']
        elif isinstance(json_resp, list):
             jobs_list = json_resp
             
        if not jobs_list or len(jobs_list) == 0:
            print("⚠️ API response success kani jobs emi levu.")
            return None
            
        print(f"📥 JSearch nundi {len(jobs_list)} raw jobs vachayi. Hashing processing start...")
        
        # 2. PREPARE DATA FOR DELTA TABLE & CREATE HASH
        records = []
        for job in jobs_list:
            if not isinstance(job, dict): continue
            
            company_name = job.get("employer_name", "Unknown Company")
            job_title = job.get("job_title", "Unknown Title")
            
            # 💡 THE HASH LOGIC: Company + Title ని కలిపి ఎన్క్రిప్ట్ చేస్తున్నాం 
            unique_string = f"{company_name}_{job_title}".lower().replace(" ", "")
            job_hash = hashlib.md5(unique_string.encode('utf-8')).hexdigest()
            
            records.append({
                "job_hash": job_hash,
                "raw_job_id": job.get("job_id", str(uuid.uuid4())),
                "fetch_date": date.today(),
                "search_keyword": search_keyword,
                "company_name": company_name,
                "job_title": job_title,
                "job_description": job.get("job_description", ""),
                "apply_link": job.get("job_apply_link", ""),
                "validation_status": "Pending" 
            })
            
        # 3. CREATE SPARK DATAFRAME WITH EXACT COLUMN ORDER
        new_jobs_df = spark.createDataFrame(records)
        ordered_columns = [
            "job_hash",
            "raw_job_id", 
            "fetch_date", 
            "search_keyword", 
            "company_name", 
            "job_title", 
            "job_description", 
            "apply_link", 
            "validation_status"
        ]
        new_jobs_df = new_jobs_df.select(*ordered_columns)
        
        # 4. 💡 DELTA MERGE LOGIC (Avoid Duplicates)
        print("🔄 Performing Delta MERGE to avoid duplicate jobs...")
        
        delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
        
        (delta_table.alias("target")
         .merge(
             new_jobs_df.alias("source"),
             "target.job_hash = source.job_hash"
         )
         .whenNotMatchedInsertAll()
         .execute())
        
        print(f"✅ Success! MERGE operation complete. Only completely NEW jobs were added.")
        
        # Save ayina data ni malli return chestunnam
        return spark.sql(check_query)
        
    else:
        print(f"❌ API Error: {response.text}")
        return None

# ==========================================
# TEST THE SMART FETCH & MERGE
# ==========================================

print("--- RUN 1 (User 1 - Python Developer) ---")
df_jobs = fetch_and_store_jobs("Python Developer in USA")
if df_jobs:
    display(df_jobs.limit(3))

print("\n--- RUN 2 (User 2 - Same day, Same keyword) ---")
df_jobs_cached = fetch_and_store_jobs("Data Engineer Jobs in USA")

In [0]:
%sql
select * from jobs_automation_db.default.raw_jobs_staging